# Perception as One Continuum: A Pre-Specified Test of Discrete Degrees in Simultaneous Order

**Registered:** 2026-10-06, before any analysis in this notebook was run. The plan cell below was committed on its own (commit message "Register notebook 09 …") and must not be edited after results exist; any later change is listed under *Deviations*.

## Why this notebook exists

Notebook 06 tested the doctrine of discrete degrees by assigning each perception marker to **one** degree (natural: vividness, memory; spiritual: reality certainty, time, thought speed; celestial: comparative reality, telepathy) and predicting (H1) a prevalence ordering natural > spiritual > celestial and (H3) degree-aligned factors. Both failed, and both were recorded as framework misses.

That operationalisation is not Swedenborg's. In *Divine Love and Wisdom* the degrees are present **together** in every created thing: "every least bit of desire, every least bit of thought, even every least bit of a mental image, is made up of levels of both kinds" (§§222–229; NCE tr. Dole). In a "simultaneous arrangement" the highest level is the centre and the lowest the circumference (§§205–208). And the earthly mind — the mind that later narrates an NDE — "is a continuum", enlightened from within by the higher levels "gradually", so that "this enlightenment of our earthly minds does not happen by distinct levels. There is instead a gradual increase … with greater clarity and purity as it ascends" (§256). No perception marker can therefore be a "celestial" or "natural" perception, and separate per-degree factors are not predicted.

## The prediction (stated before analysis)

**P1 — one cumulative continuum.** The seven perception markers of notebook 06, *unchanged* (same definitions; no item dropped or added after the fact), form a single cumulative scale of depth: accounts that report a less common marker tend also to report the more common ones. The measurement model for "a gradual increase … with greater clarity as it ascends" is a monotone-homogeneity (Mokken) scale.

*This is our operationalisation of §256, not a quotation of it.* A cumulative ordering is the simplest reading of "gradual increase"; the doctrine does not say which markers are "earlier" on the continuum, so no item order is predicted.

**Not a prediction / not counted:** that the markers form one factor (already observed in notebook 06), the absence of degree-aligned factors (already observed), and the Being-of-Light association (already observed). Those results were seen before this registration and are not evidence here.

## Analysis plan

Data: `nd.load_frame()` (N = 6,751). Items: the seven binary markers exactly as in notebook 06, cell 4/6.

1. **Scalability.** Loevinger/Mokken coefficients: pairwise $H_{ij}$, item $H_i$, scale $H$ (sum of item-pair covariances / sum of maximum covariances given the marginals). 95% CI by 1,000 bootstrap resamples of accounts.
2. **Narrative-length control (primary).** Longer accounts mention more of everything (notebook 06: length explains ~41% of the score), which inflates every covariance. The **primary statistic is the length-stratified H**: covariances and maximum covariances are computed within quintiles of log word count and pooled across quintiles. Raw H is secondary.
3. **Dimensionality.** Automated item selection (AISP, lower bound c = 0.30) on the length-stratified pairwise coefficients.
4. **Monotonicity.** Manifest monotonicity: for each item, P(item = 1) across rest-score groups (minimum group size 100); a violation is a decrease > 0.03 that is significant at α = 0.05 (one-sided z-test, Holm across items).
5. **Replication across archives.** Steps 1–2 separately in NDERF and IANDS.
6. **Reliability.** Cite inter-coder κ for each item from notebook 08; flag convention-dependent items.

## Decision rule (what counts as a hit or a miss)

| Outcome | Criterion |
|---|---|
| **Hit** | Length-stratified H ≥ 0.30 **and** AISP returns one scale containing at least 5 of the 7 items **and** no item has a significant monotonicity violation **and** both archives give stratified H ≥ 0.30 |
| **Miss** | Length-stratified H < 0.20, **or** AISP returns two or more scales of ≥ 2 items each, **or** the archives disagree (one ≥ 0.30, the other < 0.20) |
| **Underdetermined** | Anything else (e.g. stratified H between 0.20 and 0.30; a single weak scale) |

Thresholds follow Mokken's conventions (H < 0.3 is not a scale; 0.3–0.4 weak; 0.4–0.5 medium; ≥ 0.5 strong).

## What this test cannot do

- It uses the same accounts that notebook 06 analysed. Registration protects against choosing the analysis after seeing it, not against the data being familiar. A confirmatory run belongs on accounts submitted after the registration date (new NDERF/IANDS submissions), with this notebook unchanged.
- Every marker is a coded feature of one retrospective narrative (same-source caveat). A cumulative scale of *reports* is what can be shown.
- A hit is consistent with the doctrine; it does not discriminate it from any account that predicts one continuous dimension of perceptual depth. A miss counts against this operationalisation of §256.


## Setup

Items exactly as in notebook 06 (cells 4 and 6).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

NDE_ROOT = Path.cwd().parent
sys.path.insert(0, str(NDE_ROOT / "scripts"))
import nde_dataset as nd

df = nd.load_frame()
N = len(df)
print(f"Loaded {N:,} records")

PERCEPTION_MARKERS = {  # identical to notebook 06
    "sensory_vividness": ["more_vivid", "incredibly_more_vivid"],
    "memory_persistence": ["more_vivid_than_normal"],
    "reality_assessment": ["definitely_real"],
    "time_perception": ["timeless", "everything_at_once"],
    "thought_speed": ["incredibly_fast", "faster_than_normal"],
    "comparative_reality": ["more_real_explicit", "more_real_implied"],
}
X = pd.DataFrame({k: nd.isin(df, k, v).astype(int) for k, v in PERCEPTION_MARKERS.items()})
X["telepathic"] = nd.has(df, "communication_modes", "telepathic").astype(int)
items = list(X.columns)
print(X.mean().round(3).sort_values(ascending=False).to_string())

Loaded 6,751 records
reality_assessment     0.582
sensory_vividness      0.325
memory_persistence     0.289
telepathic             0.270
time_perception        0.218
comparative_reality    0.161
thought_speed          0.149


## Scalability coefficients

$H_{ij} = \mathrm{cov}_{ij} / \mathrm{cov}^{max}_{ij}$ with $\mathrm{cov}^{max}_{ij} = \min(p_i, p_j) - p_i p_j$ for binary items. Stratified versions sum the stratum-size-weighted covariances and maximum covariances within log-length quintiles.

In [2]:
def pair_sums(A):
    """Return (cov, covmax) k x k matrices for a binary array A (n x k)."""
    p = A.mean(0)
    pij = (A.T @ A) / len(A)
    cov = pij - np.outer(p, p)
    covmax = np.minimum.outer(p, p) - np.outer(p, p)
    return cov, covmax

def strata_sums(A, strata):
    cov = np.zeros((A.shape[1],) * 2); cmax = np.zeros_like(cov)
    for s in np.unique(strata):
        B = A[strata == s]
        c, m = pair_sums(B)
        cov += len(B) * c; cmax += len(B) * m
    return cov, cmax

def coefficients(cov, cmax, idx=None):
    k = cov.shape[0]
    idx = list(range(k)) if idx is None else list(idx)
    sub_c = cov[np.ix_(idx, idx)].copy(); sub_m = cmax[np.ix_(idx, idx)].copy()
    np.fill_diagonal(sub_c, 0); np.fill_diagonal(sub_m, 0)
    H = sub_c.sum() / sub_m.sum()
    Hi = sub_c.sum(1) / sub_m.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        Hij = sub_c / sub_m
    return H, Hi, Hij

A = X.values
strata = pd.qcut(df["log_words"], 5, labels=False).values

raw = coefficients(*pair_sums(A))
strat = coefficients(*strata_sums(A, strata))

rng = np.random.default_rng(20261006)
boot_raw, boot_strat = [], []
for _ in range(1000):
    ix = rng.integers(0, N, N)
    boot_raw.append(coefficients(*pair_sums(A[ix]))[0])
    boot_strat.append(coefficients(*strata_sums(A[ix], strata[ix]))[0])
ci = lambda b: tuple(np.percentile(b, [2.5, 97.5]).round(3))

print(f"Raw scale H                 = {raw[0]:.3f}  95% CI {ci(boot_raw)}")
print(f"Length-stratified scale H   = {strat[0]:.3f}  95% CI {ci(boot_strat)}   <- primary")
print("\nItem H_i (raw / length-stratified):")
print(pd.DataFrame({"raw Hi": raw[1], "stratified Hi": strat[1]}, index=items).round(3).to_string())
print("\nLength-stratified H_ij:")
print(pd.DataFrame(strat[2], index=items, columns=items).round(2).to_string())
print("\nStratified H within each length quintile:")
for s in range(5):
    print(f"  quintile {s+1}: H = {coefficients(*pair_sums(A[strata == s]))[0]:.3f}  (n = {(strata == s).sum()})")

Raw scale H                 = 0.422  95% CI (np.float64(0.408), np.float64(0.437))
Length-stratified scale H   = 0.245  95% CI (np.float64(0.231), np.float64(0.259))   <- primary

Item H_i (raw / length-stratified):
                     raw Hi  stratified Hi
sensory_vividness     0.471          0.305
memory_persistence    0.384          0.219
reality_assessment    0.663          0.431
time_perception       0.393          0.203
thought_speed         0.450          0.270
comparative_reality   0.468          0.336
telepathic            0.224          0.071

Length-stratified H_ij:
                     sensory_vividness  memory_persistence  reality_assessment  time_perception  thought_speed  comparative_reality  telepathic
sensory_vividness                  NaN                0.27                0.43             0.23           0.57                 0.59        0.03
memory_persistence                0.27                 NaN                0.63             0.14           0.19                 

## Dimensionality: automated item selection (c = 0.30, length-stratified coefficients)

In [3]:
def aisp(cov, cmax, c=0.30):
    remaining = list(range(cov.shape[0])); scales = []
    while len(remaining) >= 2:
        _, _, Hij = coefficients(cov, cmax)
        best = max(((Hij[i, j], i, j) for a, i in enumerate(remaining) for j in remaining[a + 1:]), default=None)
        if best is None or best[0] < c:
            break
        scale = [best[1], best[2]]
        while True:
            cand = []
            for k in remaining:
                if k in scale:
                    continue
                H, Hi, _ = coefficients(cov, cmax, scale + [k])
                if Hi.min() >= c:
                    cand.append((H, k))
            if not cand:
                break
            scale.append(max(cand)[1])
        scales.append(scale)
        remaining = [k for k in remaining if k not in scale]
    return scales, remaining

cov_s, cmax_s = strata_sums(A, strata)
scales, unscalable = aisp(cov_s, cmax_s)
for n_, s in enumerate(scales, 1):
    H, Hi, _ = coefficients(cov_s, cmax_s, s)
    print(f"Scale {n_}: {[items[i] for i in s]}  H = {H:.3f}  min Hi = {Hi.min():.3f}")
print("Unscalable at c = 0.30:", [items[i] for i in unscalable])

raw_scales, raw_un = aisp(*pair_sums(A))
print("\n(Secondary, raw coefficients) scales:", [[items[i] for i in s] for s in raw_scales], "unscalable:", [items[i] for i in raw_un])

Scale 1: ['reality_assessment', 'comparative_reality', 'sensory_vividness', 'thought_speed', 'memory_persistence']  H = 0.402  min Hi = 0.324
Unscalable at c = 0.30: ['time_perception', 'telepathic']

(Secondary, raw coefficients) scales: [['reality_assessment', 'comparative_reality', 'sensory_vividness', 'thought_speed', 'memory_persistence', 'time_perception']] unscalable: ['telepathic']


## Manifest monotonicity (rest-score groups, minimum size 100, minvi 0.03, one-sided z, Holm across items)

In [4]:
rows = []
for j, it in enumerate(items):
    rest = A.sum(1) - A[:, j]
    # merge adjacent rest scores into groups of >= 100
    groups, cur = [], []
    for r in range(int(rest.max()) + 1):
        cur.append(r)
        if (np.isin(rest, cur)).sum() >= 100:
            groups.append(cur); cur = []
    if cur:
        groups[-1] += cur
    props = [(A[np.isin(rest, g), j].mean(), np.isin(rest, g).sum()) for g in groups]
    worst_p, n_viol = 1.0, 0
    for a in range(len(props)):
        for b in range(a + 1, len(props)):
            (p1, n1), (p2, n2) = props[a], props[b]
            if p1 - p2 > 0.03:
                n_viol += 1
                pp = (p1 * n1 + p2 * n2) / (n1 + n2)
                z = (p1 - p2) / np.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2))
                worst_p = min(worst_p, stats.norm.sf(z))
    rows.append({"item": it, "groups": len(groups),
                 "P(item) by rest-score group": " ".join(f"{p:.2f}" for p, _ in props),
                 "violations > 0.03": n_viol, "smallest p": worst_p})
mono = pd.DataFrame(rows)
order = np.argsort(mono["smallest p"].values); m = len(mono); holm = np.empty(m); run = 0
for rank, i in enumerate(order):
    run = max(run, min(1, (m - rank) * mono["smallest p"].values[i])); holm[i] = run
mono["Holm p"] = holm
mono["significant violation"] = mono["Holm p"] < 0.05
print(mono.to_string(index=False))

               item  groups        P(item) by rest-score group  violations > 0.03  smallest p  Holm p  significant violation
  sensory_vividness       7 0.08 0.18 0.35 0.57 0.75 0.91 0.96                  0         1.0     1.0                  False
 memory_persistence       7 0.04 0.22 0.36 0.48 0.55 0.68 0.80                  0         1.0     1.0                  False
 reality_assessment       7 0.29 0.57 0.81 0.93 0.95 0.99 0.99                  0         1.0     1.0                  False
    time_perception       7 0.05 0.10 0.21 0.33 0.44 0.68 0.82                  0         1.0     1.0                  False
      thought_speed       7 0.02 0.04 0.11 0.23 0.33 0.53 0.72                  0         1.0     1.0                  False
comparative_reality       7 0.00 0.04 0.14 0.26 0.36 0.58 0.79                  0         1.0     1.0                  False
         telepathic       7 0.15 0.24 0.29 0.35 0.36 0.49 0.64                  0         1.0     1.0                  False


## Replication across archives

In [5]:
arch = {}
for ds in ["nderf", "iands"]:
    m_ = (df["dataset"] == ds).values
    Ad, sd = A[m_], pd.qcut(df.loc[m_, "log_words"], 5, labels=False).values
    Hr = coefficients(*pair_sums(Ad))[0]; Hs = coefficients(*strata_sums(Ad, sd))[0]
    arch[ds] = Hs
    print(f"{ds}: n = {m_.sum():,}  raw H = {Hr:.3f}  length-stratified H = {Hs:.3f}")

nderf: n = 5,659  raw H = 0.441  length-stratified H = 0.251
iands: n = 1,092  raw H = 0.326  length-stratified H = 0.233


## Measurement reliability of the items

From notebook 08 (blind second coder, 100 accounts): telepathic communication κ = 0.88; "more real than earthly life" (comparative reality) κ = 0.74, GPT-5.2 test–retest 0.58. Sensory vividness, memory persistence, reality certainty, time perception and thought speed were **not** among the second-coded fields; their reliability is unmeasured.

## Verdict (computed by the registered rule)

In [6]:
Hs = strat[0]
one_scale = len(scales) == 1 and len(scales[0]) >= 5
multi = sum(len(s) >= 2 for s in scales) >= 2
no_viol = not mono["significant violation"].any()
both_ok = all(v >= 0.30 for v in arch.values())
disagree = (max(arch.values()) >= 0.30) and (min(arch.values()) < 0.20)
if Hs >= 0.30 and one_scale and no_viol and both_ok:
    verdict = "HIT"
elif Hs < 0.20 or multi or disagree:
    verdict = "MISS"
else:
    verdict = "UNDERDETERMINED"
print(f"length-stratified H = {Hs:.3f}; AISP scales = {[len(s) for s in scales]}; "
      f"significant monotonicity violations = {int(mono['significant violation'].sum())}; archives = "
      + ", ".join(f"{k} {v:.3f}" for k, v in arch.items()))
print("VERDICT:", verdict)

length-stratified H = 0.245; AISP scales = [5]; significant monotonicity violations = 0; archives = nderf 0.251, iands 0.233
VERDICT: UNDERDETERMINED


## Result and interpretation

**Registered verdict: Underdetermined** (computed above by the rule fixed in the first cell; no deviation from the plan).

| Registered criterion | Result |
|---|---|
| Length-stratified H ≥ 0.30 (primary) | **0.245** (95% CI 0.231–0.259): below a scale, above the miss line (0.20) |
| AISP: one scale with ≥ 5 of 7 items | **Met**: one scale of 5 items (reality certainty, comparative reality, sensory vividness, thought speed, memory persistence; H = 0.40, min Hᵢ = 0.32); time perception and telepathy unscalable at c = 0.30 |
| No significant monotonicity violation | **Met**: no decrease > 0.03 for any item |
| Both archives ≥ 0.30 | **Not met**: NDERF 0.251, IANDS 0.233 — but the archives agree with each other |

**What the data show (statistically supported).**
- The seven markers do not split into separate scales. Five of them form one cumulative, monotone continuum after narrative length is controlled. Accounts that report a rarer marker also tend to report the commoner ones, and the pattern is the same in both archives.
- Across all seven markers, the cumulative structure is weak once length is controlled (H 0.245). Raw H is 0.42 (a medium-strength scale), but about 40% of that comes from narrative length.
- Scalability rises with narrative length: H is 0.16 in the shortest fifth of accounts and 0.30 in the longest. Two readings fit this equally well, and the data cannot separate them. Fuller narratives may reveal a continuum that short ones are too brief to show, or the structure in long accounts may be partly descriptive completeness.
- Telepathic communication does not scale with the other markers once length is controlled (Hᵢ 0.07). Time perception scales only weakly (0.20). This matches notebook 06's length-controlled correlations.

**What this means for the doctrine (interpretation).** Notebook 06's "miss" tested a structure the doctrine does not posit: one degree per marker. Under the doctrine's own description, as operationalised here and registered in advance, the result is neither a hit nor a miss. The markers behave as one continuum and not as three layers. Whether that continuum is strong enough to count as a scale is underdetermined for the seven pre-specified items. The five-item scale was selected on these data, so its H = 0.40 is optimistic and is not a test result.

**Post hoc (not evidence).** The two items that fail to scale, telepathy and time perception, are the ones whose relation to "depth" is least obviously gradual. That is an observation for the next registration, not a finding.

**Next.** (1) A confirmatory run of this notebook, unchanged, on accounts submitted after 2026-10-06. (2) Second-code sensory vividness, memory persistence, reality certainty, time perception and thought speed; their reliability is unmeasured. (3) Register any revised item set before testing it.
